In [1]:
import os

INPUT_ROOT = "/kaggle/input"

print("Input directory exists:", os.path.exists(INPUT_ROOT))

for root, dirs, files in os.walk(INPUT_ROOT):
    level = root.replace(INPUT_ROOT, "").count(os.sep)

    if level > 3:
        dirs[:] = []
        continue

    print("  " * level + os.path.basename(root) + "/")

    if level >= 2:
        dirs[:] = []

Input directory exists: True
input/
  datasets/
    aashishdev/


In [2]:
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/dataset1-clean/dataset1_clean")

TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "valid"
TEST_DIR = DATA_ROOT / "test"

print("Train exists:", TRAIN_DIR.is_dir())
print("Valid exists:", VAL_DIR.is_dir())
print("Test exists:", TEST_DIR.is_dir())

Train exists: False
Valid exists: False
Test exists: False


In [3]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.replace("/kaggle/input", "").count(os.sep)

    if depth > 4:
        dirs[:] = []
        continue

    print(root)
    for d in dirs:
        print("   DIR:", d)

/kaggle/input
   DIR: datasets
/kaggle/input/datasets
   DIR: aashishdev
/kaggle/input/datasets/aashishdev
   DIR: emotions
/kaggle/input/datasets/aashishdev/emotions
   DIR: dataset1_clean
/kaggle/input/datasets/aashishdev/emotions/dataset1_clean
   DIR: test
   DIR: train
   DIR: valid


In [4]:

from pathlib import Path

DATA_ROOT = Path(
    "/kaggle/input/datasets/aashishdev/"
    "emotions/dataset1_clean"
)

TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "valid"
TEST_DIR = DATA_ROOT / "test"

print("Train exists:", TRAIN_DIR.is_dir())
print("Validation exists:", VAL_DIR.is_dir())
print("Test exists:", TEST_DIR.is_dir())


Train exists: True
Validation exists: True
Test exists: True


In [5]:

from torchvision import datasets

CLASS_NAMES = [
    "angry", "confused", "disgusted",
    "fear", "happy", "neutral",
    "sad", "shy", "surprised"
]

train_ds = datasets.ImageFolder(TRAIN_DIR)
val_ds = datasets.ImageFolder(VAL_DIR)
test_ds = datasets.ImageFolder(TEST_DIR)

print("Classes:", train_ds.classes)
print("Train images:", len(train_ds))
print("Validation images:", len(val_ds))
print("Test images:", len(test_ds))

assert train_ds.classes == CLASS_NAMES
assert val_ds.class_to_idx == train_ds.class_to_idx
assert test_ds.class_to_idx == train_ds.class_to_idx

print("Dataset verified successfully!")


Classes: ['angry', 'confused', 'disgusted', 'fear', 'happy', 'neutral', 'sad', 'shy', 'surprised']
Train images: 121145
Validation images: 6931
Test images: 3463
Dataset verified successfully!


In [6]:

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda")
BATCH_SIZE = 32
NUM_WORKERS = 2

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(8),
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

train_ds = datasets.ImageFolder(
    TRAIN_DIR, transform=train_transform
)
val_ds = datasets.ImageFolder(
    VAL_DIR, transform=eval_transform
)
test_ds = datasets.ImageFolder(
    TEST_DIR, transform=eval_transform
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=True, persistent_workers=True
)

val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=True, persistent_workers=True
)

test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=True, persistent_workers=True
)

print("GPU:", torch.cuda.get_device_name(0))
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))


GPU: Tesla T4
Train batches: 3786
Validation batches: 217
Test batches: 109


In [7]:

import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

device = torch.device("cuda")

# Load ImageNet pretrained EfficientNet-B0
model = efficientnet_b0(
    weights=EfficientNet_B0_Weights.DEFAULT
)

# Freeze backbone for Phase 1
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier with 9-class output
in_features = model.classifier[1].in_features

model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, 9)
)

model = model.to(device)

print("Model:", "EfficientNet-B0")
print("Device:", next(model.parameters()).device)
print("Output classes:", model.classifier[1].out_features)
print("GPU:", torch.cuda.get_device_name(0))


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 152MB/s]


Model: EfficientNet-B0
Device: cuda:0
Output classes: 9
GPU: Tesla T4


In [8]:

import copy
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler("cuda")

OUTPUT_DIR = "/kaggle/working/emotion_model"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    y_true, y_pred = [], []

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(images)
                loss = criterion(logits, labels)

            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * images.size(0)
        y_pred.extend(logits.argmax(1).detach().cpu().tolist())
        y_true.extend(labels.detach().cpu().tolist())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true, y_pred,
            labels=list(range(9)),
            average="macro",
            zero_division=0
        )
    }

def fit_phase(model, train_loader, val_loader,
              checkpoint_path, epochs, lr, patience=3):

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=1
    )

    best_f1 = -1.0
    best_state = None
    stale = 0

    for epoch in range(epochs):
        train_metrics = run_epoch(model, train_loader, optimizer)
        val_metrics = run_epoch(model, val_loader)

        scheduler.step(val_metrics["macro_f1"])

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train loss: {train_metrics['loss']:.4f} | "
            f"Train acc: {train_metrics['accuracy']:.4f} | "
            f"Val loss: {val_metrics['loss']:.4f} | "
            f"Val acc: {val_metrics['accuracy']:.4f} | "
            f"Val macro-F1: {val_metrics['macro_f1']:.4f}"
        )

        if val_metrics["macro_f1"] > best_f1:
            best_f1 = val_metrics["macro_f1"]
            best_state = copy.deepcopy(model.state_dict())

            torch.save({
                "model_state_dict": best_state,
                "class_names": train_ds.classes,
                "best_val_macro_f1": best_f1
            }, checkpoint_path)

            stale = 0
        else:
            stale += 1

        if stale >= patience:
            print("Early stopping.")
            break

    model.load_state_dict(best_state)
    return model

PHASE1_PATH = f"{OUTPUT_DIR}/phase1_head.pth"

model = fit_phase(
    model,
    train_loader,
    val_loader,
    checkpoint_path=PHASE1_PATH,
    epochs=5,
    lr=3e-4,
    patience=3
)

print("Phase 1 complete. Best checkpoint:", PHASE1_PATH)


  0%|          | 0/3786 [00:00<?, ?it/s]

  0%|          | 0/217 [00:00<?, ?it/s]

Epoch 1/5 | Train loss: 1.7164 | Train acc: 0.3311 | Val loss: 1.6614 | Val acc: 0.3481 | Val macro-F1: 0.2745


  0%|          | 0/3786 [00:00<?, ?it/s]

  0%|          | 0/217 [00:00<?, ?it/s]

Epoch 2/5 | Train loss: 1.6610 | Train acc: 0.3522 | Val loss: 1.6432 | Val acc: 0.3538 | Val macro-F1: 0.2927


... (5 outputs left out)

Epoch 4/5 | Train loss: 1.6470 | Train acc: 0.3570 | Val loss: 1.6252 | Val acc: 0.3659 | Val macro-F1: 0.3012


  0%|          | 0/3786 [00:00<?, ?it/s]

  0%|          | 0/217 [00:00<?, ?it/s]

Epoch 5/5 | Train loss: 1.6471 | Train acc: 0.3584 | Val loss: 1.6160 | Val acc: 0.3715 | Val macro-F1: 0.3056
Phase 1 complete. Best checkpoint: /kaggle/working/emotion_model/phase1_head.pth


In [9]:

# Phase 2: Full fine-tuning

import torch
import os

PHASE1_PATH = "/kaggle/working/emotion_model/phase1_head.pth"
PHASE2_PATH = "/kaggle/working/emotion_model/phase2_finetuned.pth"

# Ensure Phase 1 ka best checkpoint load ho
checkpoint1 = torch.load(
    PHASE1_PATH,
    map_location=device,
    weights_only=False
)
model.load_state_dict(checkpoint1["model_state_dict"])

# Unfreeze the complete backbone
for param in model.features.parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

print("Trainable parameters enabled.")
print("Starting fine-tuning on:", torch.cuda.get_device_name(0))

model, history2 = fit_phase(
    model,
    train_loader,
    val_loader,
    checkpoint_path=PHASE2_PATH,
    epochs=8,
    lr=1e-5,
    patience=3
)

history2.to_csv(
    "/kaggle/working/emotion_model/phase2_history.csv",
    index=False
)

print("Fine-tuning complete.")
print("Checkpoint:", PHASE2_PATH)


Trainable parameters enabled.
Starting fine-tuning on: Tesla T4


  0%|          | 0/3786 [00:00<?, ?it/s]

  0%|          | 0/217 [00:00<?, ?it/s]

Epoch 1/8 | Train loss: 1.5360 | Train acc: 0.4055 | Val loss: 1.4258 | Val acc: 0.4473 | Val macro-F1: 0.3944


  0%|          | 0/3786 [00:00<?, ?it/s]

  0%|          | 0/217 [00:00<?, ?it/s]

... (16 outputs left out)

  0%|          | 0/3786 [00:00<?, ?it/s]

  0%|          | 0/217 [00:00<?, ?it/s]

Epoch 8/8 | Train loss: 1.0007 | Train acc: 0.6268 | Val loss: 1.1897 | Val acc: 0.5699 | Val macro-F1: 0.5294


TypeError: cannot unpack non-iterable EfficientNet object

In [ ]:

import os
import torch

PHASE1_PATH = "/kaggle/working/emotion_model/phase1_head.pth"
PHASE2_PATH = "/kaggle/working/emotion_model/phase2_finetuned.pth"

for path in [PHASE1_PATH, PHASE2_PATH]:
    if os.path.isfile(path):
        ckpt = torch.load(
            path, map_location="cpu", weights_only=False
        )
        print(
            os.path.basename(path),
            "| Exists: True",
            "| Best validation macro-F1:",
            ckpt.get("best_val_macro_f1", "Not recorded")
        )
    else:
        print(os.path.basename(path), "| EXISTS: False")


In [ ]:

phase1 = torch.load(
    PHASE1_PATH, map_location=device, weights_only=False
)
phase2 = torch.load(
    PHASE2_PATH, map_location=device, weights_only=False
)

if phase2["best_val_macro_f1"] > phase1["best_val_macro_f1"]:
    best_checkpoint = phase2
    best_phase = "Phase 2 - Fine-tuning"
else:
    best_checkpoint = phase1
    best_phase = "Phase 1 - Frozen backbone"

model.load_state_dict(best_checkpoint["model_state_dict"])
model.eval()

FINAL_PATH = "/kaggle/working/emotion_model/final_emotion_model.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": CLASS_NAMES,
    "class_to_idx": train_ds.class_to_idx,
    "architecture": "efficientnet_b0",
    "best_val_macro_f1": best_checkpoint["best_val_macro_f1"],
    "selected_phase": best_phase
}, FINAL_PATH)

print("Selected:", best_phase)
print("Best validation macro-F1:",
      best_checkpoint["best_val_macro_f1"])
print("Final model:", FINAL_PATH)


In [ ]:

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

FINAL_PATH = (
    "/kaggle/working/emotion_model/final_emotion_model.pth"
)

checkpoint = torch.load(
    FINAL_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

y_true, y_pred = [], []

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(images)

        predictions = logits.argmax(dim=1).cpu().numpy()

        y_pred.extend(predictions)
        y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Test images:", len(y_true))
print(f"Test accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(
    f"Test macro-F1: "
    f"{f1_score(y_true, y_pred, average='macro'):.4f}"
)

print("\nClassification Report:\n")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

cm = confusion_matrix(
    y_true, y_pred,
    labels=list(range(len(CLASS_NAMES)))
)

plt.figure(figsize=(11, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.xlabel("Predicted emotion")
plt.ylabel("Actual emotion")
plt.title("Final Test Confusion Matrix")
plt.tight_layout()
plt.savefig(
    "/kaggle/working/emotion_model/confusion_matrix.png",
    dpi=200
)
plt.show()


In [ ]:
from collections import Counter
import pandas as pd

counts = Counter(train_ds.targets)

distribution = pd.DataFrame({
    "emotion": train_ds.classes,
    "train_images": [
        counts[train_ds.class_to_idx[c]]
        for c in train_ds.classes
    ]
})

distribution["percentage"] = (
    distribution["train_images"]
    / len(train_ds) * 100
)

print(distribution.to_string(index=False))

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0

device = torch.device("cuda")

FINAL_PATH = (
    "/kaggle/working/emotion_model/final_emotion_model.pth"
)
WEIGHTED_PATH = (
    "/kaggle/working/emotion_model/weighted_finetuned.pth"
)

checkpoint = torch.load(
    FINAL_PATH,
    map_location=device,
    weights_only=False
)

model = efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features, 9
)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)

# Class counts in the same order as ImageFolder labels
counts = torch.bincount(
    torch.tensor(train_ds.targets),
    minlength=len(train_ds.classes)
).float()

# Moderate inverse-frequency weighting
class_weights = (counts.sum() / (len(counts) * counts)).sqrt()
class_weights = class_weights / class_weights.mean()
class_weights = class_weights.to(device)

print("Classes:", train_ds.classes)
print("Counts:", counts.tolist())
print("Weights:", class_weights.cpu().tolist())

criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
# Fine-tune the current best model with a lower learning rate
for param in model.features.parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

model, weighted_history = fit_phase(
    model,
    train_loader,
    val_loader,
    checkpoint_path=WEIGHTED_PATH,
    epochs=5,
    lr=5e-6,
    patience=2
)

print("Weighted experiment complete:", WEIGHTED_PATH)

In [ ]:

import os
import torch

WEIGHTED_PATH = (
    "/kaggle/working/emotion_model/weighted_finetuned.pth"
)

if os.path.exists(WEIGHTED_PATH):
    ckpt = torch.load(
        WEIGHTED_PATH,
        map_location="cpu",
        weights_only=False
    )
    print("Checkpoint exists: True")
    print("Best validation macro-F1:",
          ckpt["best_val_macro_f1"])
else:
    print("Checkpoint exists: False")


In [ ]:

import os

OUTPUT_DIR = "/kaggle/working/emotion_model"

for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if file.endswith(".pth"):
            print(os.path.join(root, file))


In [ ]:

import torch
import os

WEIGHTED_PATH = (
    "/kaggle/working/emotion_model/weighted_finetuned.pth"
)

if os.path.exists(WEIGHTED_PATH):
    ckpt = torch.load(
        WEIGHTED_PATH,
        map_location="cpu",
        weights_only=False
    )
    print("Weighted checkpoint found.")
    print("Best validation macro-F1:",
          ckpt["best_val_macro_f1"])
else:
    print("Weighted checkpoint bhi nahi mila.")


In [ ]:
import os

for base in ["/kaggle/working", "/kaggle/input"]:
    print(f"\nSearching in {base}")

    for root, dirs, files in os.walk(base):
        for file in files:
            if file.endswith((".pth", ".pt")):
                print(os.path.join(root, file))

In [ ]:
import os
import torch

OUTPUT_DIR = "/kaggle/working/emotion_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Checkpoint directory:", OUTPUT_DIR)
print("Directory exists:", os.path.exists(OUTPUT_DIR))

In [ ]:
import os

OUTPUT_DIR = "/kaggle/working/emotion_model"

files = os.listdir(OUTPUT_DIR)

print("Files found:", len(files))

for file in files:
    path = os.path.join(OUTPUT_DIR, file)
    print(
        file,
        f"({os.path.getsize(path) / (1024**2):.2f} MB)"
        if os.path.isfile(path) else "(folder)"
    )

In [ ]:
import os
import torch

DATA_ROOT = (
    "/kaggle/input/datasets/aashishdev/"
    "emotions-clean-dataset/dataset1_clean"
)

OUTPUT_DIR = "/kaggle/working/emotion_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for split in ["train", "valid", "test"]:
    path = os.path.join(DATA_ROOT, split)
    print(split, "exists:", os.path.isdir(path))

print("GPU:", torch.cuda.get_device_name(0))
print("Output directory:", OUTPUT_DIR)

In [ ]:

import os
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

DATA_ROOT = Path(
    "/kaggle/input/datasets/aashishdev/"
    "emotions-clean-dataset/dataset1_clean"
)
OUTPUT_DIR = "/kaggle/working/emotion_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda")

CLASS_NAMES = [
    "angry", "confused", "disgusted",
    "fear", "happy", "neutral", "sad",
    "shy", "surprised"
]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(8),
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

train_ds = datasets.ImageFolder(
    DATA_ROOT / "train", transform=train_transform
)
val_ds = datasets.ImageFolder(
    DATA_ROOT / "valid", transform=eval_transform
)
test_ds = datasets.ImageFolder(
    DATA_ROOT / "test", transform=eval_transform
)

assert train_ds.classes == CLASS_NAMES
assert val_ds.class_to_idx == train_ds.class_to_idx
assert test_ds.class_to_idx == train_ds.class_to_idx

BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True
)

print("GPU:", torch.cuda.get_device_name(0))
print("Images:", len(train_ds), len(val_ds), len(test_ds))
print("Classes:", train_ds.classes)


In [ ]:

from torchvision.models import (
    efficientnet_b0, EfficientNet_B0_Weights
)

model = efficientnet_b0(
    weights=EfficientNet_B0_Weights.DEFAULT
)

for param in model.features.parameters():
    param.requires_grad = True

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features, 9
)

model = model.to(device)

# Moderate class weights from training distribution
counts = torch.bincount(
    torch.tensor(train_ds.targets),
    minlength=9
).float()

class_weights = (
    counts.sum() / (len(counts) * counts)
).sqrt()

class_weights = (
    class_weights / class_weights.mean()
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

print("Model initialized.")
print("Class weights:", class_weights.cpu().numpy())


In [ ]:

import copy
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

BEST_PATH = f"{OUTPUT_DIR}/weighted_best.pth"
LATEST_PATH = f"{OUTPUT_DIR}/weighted_latest.pth"

scaler = torch.amp.GradScaler("cuda")

def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    ys, ps = [], []

    with torch.inference_mode():
        for x, y in tqdm(loader, leave=False):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(x)
                loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)
            ps.extend(logits.argmax(1).cpu().tolist())
            ys.extend(y.cpu().tolist())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(ys, ps),
        "macro_f1": f1_score(
            ys, ps, average="macro", zero_division=0
        )
    }

def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    ys, ps = [], []

    for x, y in tqdm(loader, leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * x.size(0)
        ps.extend(logits.argmax(1).detach().cpu().tolist())
        ys.extend(y.detach().cpu().tolist())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(ys, ps)
    }

optimizer = torch.optim.AdamW(
    model.parameters(), lr=5e-6, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=1
)

best_f1 = -1.0
stale_epochs = 0
history = []

for epoch in range(5):
    tr = train_one_epoch(model, train_loader, optimizer)
    va = evaluate(model, val_loader)
    scheduler.step(va["macro_f1"])

    record = {
        "epoch": epoch + 1,
        "train_loss": tr["loss"],
        "train_accuracy": tr["accuracy"],
        "val_loss": va["loss"],
        "val_accuracy": va["accuracy"],
        "val_macro_f1": va["macro_f1"]
    }
    history.append(record)

    print(record)

    # Save latest weights every epoch
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_macro_f1": max(best_f1, va["macro_f1"]),
        "class_names": CLASS_NAMES
    }, LATEST_PATH)

    if va["macro_f1"] > best_f1:
        best_f1 = va["macro_f1"]
        stale_epochs = 0

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": copy.deepcopy(model.state_dict()),
            "best_val_macro_f1": best_f1,
            "class_names": CLASS_NAMES
        }, BEST_PATH)
    else:
        stale_epochs += 1

    if stale_epochs >= 3:
        print("Early stopping.")
        break

pd.DataFrame(history).to_csv(
    f"{OUTPUT_DIR}/weighted_history.csv", index=False
)

print("Best checkpoint:", BEST_PATH)
print("Latest checkpoint:", LATEST_PATH)


In [ ]:
import pandas as pd
import os

history_path = f"{OUTPUT_DIR}/weighted_history.csv"

pd.DataFrame(history).to_csv(
    history_path,
    index=False
)

print("History saved:", history_path)

for name in ["weighted_best.pth", "weighted_latest.pth"]:
    path = os.path.join(OUTPUT_DIR, name)
    print(name, "exists:", os.path.isfile(path))

In [ ]:
import torch

BEST_PATH = f"{OUTPUT_DIR}/weighted_best.pth"

if os.path.isfile(BEST_PATH):
    ckpt = torch.load(
        BEST_PATH,
        map_location="cpu",
        weights_only=False
    )
    print("Best epoch:", ckpt["epoch"])
    print("Best validation macro-F1:",
          ckpt["best_val_macro_f1"])
else:
    print("Best checkpoint missing!")

In [ ]:
import torch

BEST_PATH = (
    "/kaggle/working/emotion_model/weighted_best.pth"
)

checkpoint = torch.load(
    BEST_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Best epoch:", checkpoint["epoch"])
print("Validation macro-F1:",
      checkpoint["best_val_macro_f1"])

In [ ]:
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

checkpoint = torch.load(
    "/kaggle/working/emotion_model/weighted_best.pth",
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

y_true, y_pred = [], []

with torch.inference_mode():
    for images, labels in val_loader:
        images = images.to(device, non_blocking=True)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(images)

        y_pred.extend(logits.argmax(1).cpu().numpy())
        y_true.extend(labels.numpy())

print(classification_report(
    y_true,
    y_pred,
    labels=list(range(9)),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred, labels=list(range(9)))

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt="d",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Weighted Model — Validation Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(CLASS_NAMES)))
)

# Normalize each actual class row
row_totals = cm.sum(axis=1, keepdims=True)
cm_normalized = np.divide(
    cm,
    row_totals,
    out=np.zeros_like(cm, dtype=float),
    where=row_totals != 0
)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".1%",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    vmin=0,
    vmax=1
)
plt.xlabel("Predicted emotion")
plt.ylabel("Actual emotion")
plt.title("Normalized Validation Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

model.eval()

# Images with the lowest true-class probability
# among incorrectly classified examples
errors = []

with torch.inference_mode():
    for images, labels in val_loader:
        images_gpu = images.to(device)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(images_gpu)

        probs = torch.softmax(logits, dim=1).cpu()
        preds = probs.argmax(dim=1)

        for i in range(len(labels)):
            true_label = labels[i].item()
            pred_label = preds[i].item()

            if true_label != pred_label:
                errors.append({
                    "true": true_label,
                    "pred": pred_label,
                    "confidence": probs[i, pred_label].item(),
                    "true_probability": probs[i, true_label].item(),
                    "path": val_ds.samples[
                        len(errors) * 0
                    ][0] if False else None
                })

In [ ]:
errors = []

with torch.inference_mode():
    for images, labels in val_loader:
        images_gpu = images.to(device)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(images_gpu)

        probs = torch.softmax(logits, dim=1).cpu()
        preds = probs.argmax(dim=1)

        for i in range(len(labels)):
            true_label = labels[i].item()
            pred_label = preds[i].item()

            if true_label != pred_label:
                errors.append({
                    "true": true_label,
                    "pred": pred_label,
                    "confidence": probs[i, pred_label].item(),
                    "true_probability": probs[i, true_label].item()
                })

print("Misclassified images:", len(errors))

In [ ]:
import matplotlib.pyplot as plt
import random
from PIL import Image

# ImageFolder stores (path, label) pairs
samples = val_ds.samples
errors = []

model.eval()

with torch.inference_mode():
    for images, labels in val_loader:
        images_gpu = images.to(device)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(images_gpu)

        probs = torch.softmax(logits, dim=1).cpu()
        preds = probs.argmax(dim=1)

        for i, true_label in enumerate(labels.tolist()):
            pred_label = preds[i].item()

            if true_label != pred_label:
                errors.append({
                    "true": true_label,
                    "pred": pred_label,
                    "confidence": probs[i, pred_label].item()
                })

# Recover paths in the same order as the non-shuffled validation loader
error_samples = []
error_idx = 0

for sample_idx, (_, true_label) in enumerate(samples):
    # This list is built from the validation dataset's ordered predictions.
    pass

In [ ]:

import torch
import matplotlib.pyplot as plt
import random
from PIL import Image

model.eval()
error_samples = []

# Validation DataLoader shuffle=False hona chahiye
with torch.inference_mode():
    offset = 0

    for images, labels in val_loader:
        images_gpu = images.to(device, non_blocking=True)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(images_gpu)

        probs = torch.softmax(logits, dim=1).cpu()
        preds = probs.argmax(dim=1)

        for i, true_label in enumerate(labels.tolist()):
            pred_label = preds[i].item()

            if (
                true_label != pred_label
                and CLASS_NAMES[true_label] in ["fear", "sad"]
            ):
                path, _ = val_ds.samples[offset + i]

                error_samples.append({
                    "path": path,
                    "true": true_label,
                    "pred": pred_label,
                    "confidence": probs[i, pred_label].item()
                })

        offset += len(labels)

print("Fear/Sad misclassified images:", len(error_samples))

# Show 20 examples
random.seed(42)
examples = random.sample(
    error_samples,
    min(20, len(error_samples))
)

fig, axes = plt.subplots(4, 5, figsize=(16, 13))
axes = axes.flatten()

for ax, item in zip(axes, examples):
    with Image.open(item["path"]) as im:
        ax.imshow(im.convert("RGB"))

    ax.set_title(
        f"True: {CLASS_NAMES[item['true']]}\n"
        f"Pred: {CLASS_NAMES[item['pred']]}\n"
        f"Confidence: {item['confidence']:.1%}",
        fontsize=9
    )
    ax.axis("off")

for ax in axes[len(examples):]:
    ax.axis("off")

plt.tight_layout()
plt.show()
